# 5. Checking a structure

Before a structure is used as a reference (or a model is trusted), `verify` checks that it
is one complete complex: the chains are all there, they touch where they should, the TCR
sits on the pMHC, and the structure itself has no breaks or missing anchor residues. It
returns one row per check, `pass`, `warn` or `fail`, with what it found.

Three examples:

- **1AO7**, a normal complex: every check passes.
- **2E7L**, two copies of the complex in one entry, and one per RCSB biological assembly.
- **9RUP**, a cryo-EM dimer whose TCR barely touches the pMHC: caught by the
  `TCR docking` warning.

In [1]:
import pandas as pd
from pdbframe import PDBFrame

import tcrstruct as tcs
from tcrstruct import TCRFrame

pd.options.display.precision = 3

## 1. A complete complex

On 1AO7 every check passes. Besides the chains and contacts, `verify` checks chain breaks
(and the region they fall in), the conserved TCR residues (Cys23, Trp41, Cys104,
Phe/Trp118) and, for IMGT-numbered files, the numbering. On the command line:
`tcrstruct verify structures/ -o checks.csv`.

In [2]:
s = TCRFrame.read(tcs.example('1AO7', 'rcsb.pdb'), name='1AO7')
s.verify()

,check,status,detail
0,MHC class,ok,"class I: A, B2M"
1,chains,ok,"all present: A, C, D, E, F"
2,peptide,ok,9 residues
3,peptide in the MHC groove,ok,9 of 9 peptide residues touch the MHC
4,TCR on the pMHC,ok,"14 CDR residues touch the peptide, 16 touch th..."
5,TCR docking,ok,"5 of 6 CDRs on the pMHC, 14 CDR residues on th..."
6,TCR alpha-beta pairing,ok,43 residue pairs in contact
7,B2M on the heavy chain,ok,69 residue pairs in contact
8,chain breaks,warning,"3: E 141-142: 5.8 A, at least 1 missing; E 176..."
9,TCR anchors,ok,"Cys23, Trp41, Cys104, Phe/Trp118 in both V dom..."


`as_bool=True` gives one True / False, for filtering a data set:

In [3]:
s.verify(as_bool=True)

True

## 2. Several copies, and RCSB's biological assemblies

An RCSB entry is the **deposited structure** (the asymmetric unit), and it often holds two
or more copies of the complex. `TCRFrame.from_rcsb(id)` reduces it to one in two steps:

| step | argument | what it does |
|---|---|---|
| RCSB's biological assembly | `assembly=1` (default), `2`, ..., `None` = the deposited structure | what the depositors call the biological unit; usually one complex, **but not always** (9RUP below) |
| tcrstruct, by contacts | `complex=1` (default), `2`, ..., `None` = every chain | one complex: chains grouped by type and by which chains touch |

`assembly=None, complex=None` gives the whole entry as deposited.

2E7L has two copies of the 2C TCR on H-2Ld, and RCSB gives one per assembly:

In [4]:
{n: PDBFrame.from_rcsb('2E7L', assembly=n).chains for n in [1, 2]}

{1: ['A', 'D', 'E', 'Q'], 2: ['B', 'C', 'F', 'P']}

So either assembly is one complex, and passes. 2E7L has no B2M, which is a warning
(`require_b2m=True` makes it a failure):

In [5]:
second = TCRFrame.from_rcsb('2E7L', assembly=2)
second.chain_table[['role', 'original_chain', 'length']]

,role,original_chain,length
chain,,,
A,MHC heavy chain,F,175
C,peptide,P,9
D,TCR alpha,B,110
E,TCR beta,C,112


In [6]:
second.verify()

,check,status,detail
0,MHC class,ok,class I: L
1,chains,warning,B2M missing (require_b2m=True makes this a fai...
2,peptide,ok,9 residues
3,peptide in the MHC groove,ok,9 of 9 peptide residues touch the MHC
4,TCR on the pMHC,ok,"14 CDR residues touch the peptide, 14 touch th..."
5,TCR docking,ok,"6 of 6 CDRs on the pMHC, 14 CDR residues on th..."
6,TCR alpha-beta pairing,ok,48 residue pairs in contact
7,chain breaks,ok,no breaks in chains A-F
8,TCR anchors,ok,"Cys23, Trp41, Cys104, Phe/Trp118 in both V dom..."
9,TCR numbering,ok,the file is not IMGT numbered (labels are in t...


### What goes wrong with every chain

Taking the whole deposit (`assembly=None, complex=None`), each standard ID goes to the
first chain of its kind by chain ID. 2E7L's chain IDs are not in copy order, so this mixes
the copies: MHC E with peptide **P** (in the other MHC's groove), and TCR alpha A with beta
**C** (which pairs with alpha B). `verify` catches it: no peptide residue touches the
MHC, and the two TCR chains do not touch.

In [7]:
whole = TCRFrame.from_rcsb('2E7L', assembly=None, complex=None)
whole.verify(complex=None)

,check,status,detail
0,MHC class,ok,class I: L
1,chains,warning,B2M missing (require_b2m=True makes this a fai...
2,peptide,ok,9 residues
3,extra chains,warning,"4 more chains (G, H, I, J): other copies in th..."
4,one complex,fail,the chains picked mix copies: A (file E): comp...
5,peptide in the MHC groove,fail,0 of 9 peptide residues touch the MHC
6,TCR on the pMHC,ok,"9 CDR residues touch the peptide, 9 touch the MHC"
7,TCR docking,ok,"6 of 6 CDRs on the pMHC, 9 CDR residues on the..."
8,TCR alpha-beta pairing,fail,0 residue pairs in contact
9,chain breaks,ok,no breaks in chains A-F


`tcs.complex_table` groups the chains by contacts instead: each MHC chain takes the
peptide in its groove, then the TCR on that pMHC.

In [8]:
tcs.complex_table(whole)

,mhc,mhc_partner,peptide,tcr_alpha,tcr_beta,chains
complex,,,,,,
1,E,-,Q,A,D,"[E, Q, A, D]"
2,F,-,P,B,C,"[F, P, B, C]"


`tcs.find_complexes` returns each complex as its own TCRFrame; `complex=n` reads one of
them (`TCRFrame.read(path, complex=2)`, `from_rcsb(..., assembly=None, complex=2)`).

In [9]:
first, second = tcs.find_complexes(whole)
first.metadata['complex_chains'], second.metadata['complex_chains']

(['E', 'Q', 'A', 'D'], ['F', 'P', 'B', 'C'])

## 3. 9RUP: two copies in one assembly, and a TCR off the pMHC

9RUP is a cryo-EM structure of two TCR-pMHCs. The depositors call the whole dimer the
biological unit, so assembly 1 has all ten chains, and there is no assembly 2:

In [10]:
PDBFrame.from_rcsb('9RUP', assembly=1).chains

['A', 'B', 'E', 'C', 'D', 'a', 'b', 'e', 'c', 'd']

Here the second step does the work: `from_rcsb('9RUP')` keeps complex 1 by contacts, with a
warning naming the chains it left out. `complex=2` reads the lowercase copy.

In [11]:
c1 = TCRFrame.from_rcsb('9RUP')               # assembly=1, complex=1
c1.chain_table[['role', 'original_chain', 'gene', 'length']]

tcrstruct/chains.py:205: TCRStructWarning: 9RUP: 2 copies of the complex (by contacts); kept complex 1 (file chains C, E, D, A, B) and left out chains a, b, c, d, e; complex=n reads another copy, complex=None every chain, tcs.complex_table() shows how they group
  part, picked = _pick_complex(pf, table, complex, name or pf.name)


,role,original_chain,gene,length
chain,,,,
A,MHC heavy chain,C,A,260
C,peptide,D,-,9
D,TCR alpha,A,TRAV3,198
E,TCR beta,B,TRBV5-1,242
F,B2M,E,B2M,99


### Is the TCR really on the pMHC?

The chains are all there, but the TCR hardly touches the pMHC. It passes `TCR on the pMHC`
(any CDR contact counts) and gets a warning in `TCR docking`:

In [12]:
c1.verify()

,check,status,detail
0,MHC class,ok,"class I: A, B2M"
1,chains,ok,"all present: A, C, D, E, F"
2,peptide,ok,9 residues
3,copies,warning,"complex 1 of 2 in the file (by contacts), file..."
4,peptide in the MHC groove,ok,8 of 9 peptide residues touch the MHC
5,TCR on the pMHC,ok,"1 CDR residues touch the peptide, 4 touch the MHC"
6,TCR docking,warning,only 1 CDR residue on the peptide; only 3 of 6...
7,TCR alpha-beta pairing,ok,110 residue pairs in contact
8,B2M on the heavy chain,ok,60 residue pairs in contact
9,chain breaks,warning,"4: A 215-218: 10.0 A, at least 2 missing; A 25..."


`cdr_contacts` shows why, per CDR: its closest distance to the peptide and to the MHC, and
how many of its residues are within 4.5 Å of each. Next to 1AO7:

In [13]:
pd.concat({'9RUP': c1.cdr_contacts(), '1AO7': s.cdr_contacts()}, axis=1)

9RUP                                             \
      peptide_distance mhc_distance peptide_residues mhc_residues   
CDR                                                                 
CDRA1           13.666       11.654                0            0   
CDRA2            7.613        3.814                0            1   
CDRA3           13.423       11.932                0            0   
CDRB1            8.895        2.818                0            1   
CDRB2           19.779       15.352                0            0   
CDRB3            4.101        3.263                1            2   

                               1AO7                                \
      touches_pmhc peptide_distance mhc_distance peptide_residues   
CDR                                                                 
CDRA1        False            2.542        3.295                3   
CDRA2         True            6.836        3.125                0   
CDRA3        False            2.887        2.851                4   
CDRB1         True            3.351        6.878                1   
CDRB2        False            6.598        6.173                0   
CDRB3         True            3.196        2.657                6   

                                 
      mhc_residues touches_pmhc  
CDR                              
CDRA1            4         True  
CDRA2            3         True  
CDRA3            4         True  
CDRB1            0         True  
CDRB2            0        False  
CDRB3            5         True

In 9RUP only three CDRs come within 4.5 Å of the pMHC, one CDR residue touches the
peptide, and CDRA3 is 13 Å from it. Over a set of 320 RCSB TCR-pMHC structures the
median is 6 CDRs and 10 CDR residues on the peptide, and the lowest are 3 CDRs and 3
residues. `TCR docking` warns below 4 CDRs or 3 peptide residues, or for a crossing angle
over 90° (reverse docking); in those 320 it flags only 9RUP and six reverse-docked TCRs.
A warning is for checking by eye:

In [14]:
c1.view(regions=['cdrs', 'peptide_core'], tcr_up=True)

## 4. Splitting a complex

`split` gives the TCR, pMHC, MHC and peptide as separate structures, or writes them with
`out_dir=`; any region works as a part too. It takes the complex you read (on the command
line: `tcrstruct split structure.cif parts/ --complex 2`).

In [15]:
{name: part.chains for name, part in s.split(['tcr', 'pmhc', 'mhc', 'peptide', 'cdrs']).items()}

{'tcr': ['D', 'E'],
 'pmhc': ['A', 'C', 'F'],
 'mhc': ['A', 'F'],
 'peptide': ['C'],
 'cdrs': ['D', 'E']}